# Chapter 02: AI Agent Basics

Build an AI agent from scratch using the OpenAI API.

**Sections:**
- 2.0 Setup & Environment
- 2.2 Your First AI Agent (Prompt-based function selection)
- 2.3 Adding Memory (Conversation history)
- 2.4 Adding Tools (JSON Schema tool definitions)
- 2.5 Adding Function Calling (process_ai_response)

`get_currency`
`get_news`- 2.6 Tool Results (Complete agent loop)

---
## 2.0 Setup & Environment

In [4]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("OPENAI_MODEL_NAME:", os.getenv("OPENAI_MODEL_NAME"))

OPENAI_API_KEY: a703ad29...
OPENAI_BASE_URL: https://lgcns-assetization3-japan-east.openai.azure.com/openai/v1
OPENAI_MODEL_NAME: gpt-5.1


---
## 2.2 Your First AI Agent

Use prompt engineering to make the AI select a function by name.
This is the most primitive form of an AI agent — no official function calling yet.

In [10]:
import openai

client = openai.OpenAI()

PROMPT = """
I have the following functions in my system.

`get_weather`
`get_currency`
`get_news`

All of them receive the name of a country as an argument (i.e get_news('Spain'))

Please answer with the name of the function that you would like me to run.

Please say nothing else, just the name of the function with the arguments.

Answer the following question:

What is the weather in Greece?
"""

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
    messages=[{"role": "user", "content": PROMPT}],
)

response

ChatCompletion(id='chatcmpl-DNpLGuPCYbuPetV8aShBpi46PWX0r', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="get_weather('Greece')", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1774571274, model='gpt-5.1-2025-11-13', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=16, prompt_tokens=91, total_tokens=107, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [11]:
# Extract the message content
message = response.choices[0].message.content
message

"get_weather('Greece')"

### Exercise 2.2

1. Change the question to `"What is the currency of Japan?"` — does the AI return `get_currency('Japan')`?
2. Remove `"Please say nothing else"` from the prompt — how does the response change?
3. Try a different model and compare the results.

In [7]:
# Try it yourself


---
## 2.3 Adding Memory

LLMs are stateless — they don't remember previous messages unless you send the full history every time.
The `messages` array is the memory.

In [13]:
import openai

client = openai.OpenAI()
messages = []


def call_ai():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
        messages=messages,
    )
    message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": message})
    print(f"AI: {message}")

In [14]:
# Conversation loop — type 'q' to quit
while True:
    user_input = input("You: ")
    if user_input in ("quit", "q"):
        break
    messages.append({"role": "user", "content": user_input})
    print(f"User: {user_input}")
    call_ai()

User: what
AI: Can you clarify what you’d like to know or do? For example, you might be asking:

- What is [a concept]?
- What should I do about [a situation]?
- What does [term] mean?
- What are the steps to [task]?

Give me a bit more detail and I’ll help.
User: why
AI: I need a bit more context to answer.

“Why” about what? For example:
- Why does something work a certain way?
- Why did someone do something?
- Why are you experiencing a certain problem?
- Why should you choose option A over B?

Tell me the topic or situation you’re asking “why” about, and I’ll explain.
User: hello
AI: Hello!  

What would you like to talk about or work on today?
User: stop
AI: Okay, I’ll stop here. If you want to continue later, just send another message.


KeyboardInterrupt: Interrupted by user

In [15]:
# Inspect the conversation history
for m in messages:
    print(f"[{m['role']}] {m['content'][:80]}")

[user] what
[assistant] Can you clarify what you’d like to know or do? For example, you might be asking:
[user] why
[assistant] I need a bit more context to answer.

“Why” about what? For example:
- Why does 
[user] hello
[assistant] Hello!  

What would you like to talk about or work on today?
[user] stop
[assistant] Okay, I’ll stop here. If you want to continue later, just send another message.


### Exercise 2.3

1. Have a long conversation — does the AI remember everything?
2. Reset memory with `messages = []` mid-conversation — does the AI forget?
3. Add a system message to change the AI's personality:
   ```python
   messages = [{"role": "system", "content": "You are a pirate. Speak like a pirate."}]
   ```

In [ ]:
# Try it yourself


---
## 2.4 Adding Tools

Instead of prompt-based function selection, use OpenAI's official **Tools** feature.
Define functions with JSON Schema so the AI returns structured `tool_calls`.

In [16]:
import openai

client = openai.OpenAI()
messages = []


# A mock function (in production, call a real weather API)
def get_weather(city):
    return "33 degrees celcius."


FUNCTION_MAP = {"get_weather": get_weather}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "A function to get the weather of a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather of.",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

In [17]:
def call_ai():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
        messages=messages,
        tools=TOOLS,
    )
    msg = response.choices[0].message
    print("finish_reason:", response.choices[0].finish_reason)
    print("tool_calls:", msg.tool_calls)
    print("content:", msg.content)
    messages.append({"role": "assistant", "content": msg.content})
    print(f"AI: {msg.content}")

In [18]:
# Test 1: Normal conversation (finish_reason = 'stop')
messages = []
messages.append({"role": "user", "content": "My name is Nico"})
call_ai()

finish_reason: stop
tool_calls: None
content: Nice to meet you, Nico. How can I help you today?
AI: Nice to meet you, Nico. How can I help you today?


In [19]:
# Test 2: Tool-required question (finish_reason = 'tool_calls')
messages = []
messages.append({"role": "user", "content": "What is the weather in Spain?"})
call_ai()

finish_reason: tool_calls
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_3FGZn5IIg7IRZ0Ra5dq5BcfI', function=Function(arguments='{"city":"Madrid"}', name='get_weather'), type='function')]
content: None
AI: None


### Exercise 2.4

1. Add a `get_news` tool schema to the `TOOLS` list
2. Alternate between tool-needed and normal questions — observe `finish_reason`
3. Change the `description` — does the AI's tool selection behavior change?

In [ ]:
# Try it yourself


---
## 2.5 Adding Function Calling

Handle the AI's `tool_calls` response — actually execute the requested function
and append the result to the message history.

In [20]:
import openai, json

client = openai.OpenAI()
messages = []


def get_weather(city):
    return "33 degrees celcius."


FUNCTION_MAP = {"get_weather": get_weather}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "A function to get the weather of a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather of.",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

In [21]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        # Step 1: Append the assistant's tool_calls message to history
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        # Step 2: Execute each tool call
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)

            # Step 3: Append tool result to history
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")

In [22]:
def call_ai():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [23]:
# Test: The function executes but AI doesn't get the result yet
messages = []
messages.append({"role": "user", "content": "What is the weather in Spain?"})
call_ai()

print("\n--- messages history ---")
for m in messages:
    print(m)

Calling function: get_weather with {"city":"Madrid"}

--- messages history ---
{'role': 'user', 'content': 'What is the weather in Spain?'}
{'role': 'assistant', 'content': '', 'tool_calls': [{'id': 'call_rSt7cIS2UyIbdiA7lSOfynSC', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"city":"Madrid"}'}}]}
{'role': 'tool', 'tool_call_id': 'call_rSt7cIS2UyIbdiA7lSOfynSC', 'name': 'get_weather', 'content': '33 degrees celcius.'}


---
## 2.6 Tool Results — Complete Agent Loop

After executing the tool, call the AI again so it can use the result
to generate the final answer. This completes the **agent loop**:

```
User question -> AI decides -> tool_calls -> execute function
    -> append result -> call AI again -> final text response
```

In [25]:
import openai, json

client = openai.OpenAI()
messages = []


def get_weather(city):
    return "33 degrees celcius."


FUNCTION_MAP = {"get_weather": get_weather}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "A function to get the weather of a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather of.",
                    }
                },
                "required": ["city"],
            },
        },
    }
]


def process_ai_response(message):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments
            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )

        # KEY: Call AI again so it can use the tool result!
        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [ ]:
# Complete agent loop — type 'q' to quit
messages = []

while True:
    user_input = input("You: ")
    if user_input in ("quit", "q"):
        break
    messages.append({"role": "user", "content": user_input})
    print(f"User: {user_input}")
    call_ai()

In [ ]:
# Inspect the full conversation history
for i, m in enumerate(messages):
    role = m['role']
    content = m.get('content', '')[:80]
    tool_calls = 'tool_calls' in m
    print(f"[{i}] {role}: {content} {'(has tool_calls)' if tool_calls else ''}")

### Final Exercise

1. Add `get_news` and `get_currency` functions + tool schemas
2. Test multi-turn conversation: `"What's the weather in Korea?"` then `"What about the news there?"`
3. Add `print(messages)` inside `process_ai_response` to trace the full history at each step
4. What happens if the AI calls a function not in `FUNCTION_MAP`? Add error handling.

In [ ]:
# Try it yourself
